# Fine‑Tuning MobileNetV2 + SVM Pipeline
This notebook reproduces the full training, feature extraction, SVM training, and evaluation steps previously implemented in a Python script.

## Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow import keras

## Configuration

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 2

TRAIN_DIR = "../data/train"
VAL_DIR   = "../data/val"
TEST_DIR  = "../data/test"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "mobilenet_final_tf2.h5")

HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 1
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5


## Data Generators

In [4]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1
).flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = ImageDataGenerator(
    rescale=1./255
).flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = ImageDataGenerator(
    rescale=1./255
).flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 5215 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


## Build MobileNetV2 Base Model

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling="avg"
)

base_model.trainable = False  # train head first

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Train Head

In [6]:
history_head = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=HEAD_EPOCHS
)

# ====================================
# FINE-TUNE BASE MODEL
# ====================================
for layer in base_model.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=FINE_TUNE_EPOCHS
)

Epoch 1/5
163/163 [==============================] - 191s 1s/step - loss: 0.2154 - accuracy: 0.9110 - val_loss: 0.3924 - val_accuracy: 0.8125
Epoch 2/5
163/163 [==============================] - 145s 882ms/step - loss: 0.1434 - accuracy: 0.9427 - val_loss: 0.3312 - val_accuracy: 0.8125
Epoch 3/5
163/163 [==============================] - 191s 1s/step - loss: 0.1276 - accuracy: 0.9507 - val_loss: 0.4985 - val_accuracy: 0.8125
Epoch 4/5
163/163 [==============================] - 184s 1s/step - loss: 0.1332 - accuracy: 0.9480 - val_loss: 0.3978 - val_accuracy: 0.8125
Epoch 5/5
163/163 [==============================] - 177s 1s/step - loss: 0.1109 - accuracy: 0.9561 - val_loss: 0.3149 - val_accuracy: 0.8750


In [ ]:
import tensorflow as tf

tf.keras.models.save_model(
    model,
    "mobilenet_final_tf2.h5",
    save_format="h5"
)

## Save the model

In [ ]:
model.save(FINAL_MODEL_PATH)
print("Saved model to:", FINAL_MODEL_PATH)


## Visualization and metrics on performance

In [ ]:
# ====================================
# COMBINE HISTORY FOR PLOTTING
# ====================================
def combine_histories(h1, h2):
    history = {}
    for k in h1.history.keys():
        history[k] = h1.history[k] + h2.history[k]
    return history

full_history = combine_histories(history_head, history_fine)

# ====================================
# PLOT ACCURACY & LOSS
# ====================================
epochs_range = range(len(full_history["accuracy"]))

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, full_history["accuracy"], label="Train Acc")
plt.plot(epochs_range, full_history["val_accuracy"], label="Val Acc")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, full_history["loss"], label="Train Loss")
plt.plot(epochs_range, full_history["val_loss"], label="Val Loss")
plt.title("Loss")
plt.legend()

plt.show()



## Evaluate on Test Set

In [12]:
# ====================================
# TEST SET EVALUATION
# ====================================
test_preds = model.predict(test_gen)
y_pred = np.argmax(test_preds, axis=1)
y_true = test_gen.classes

class_labels = list(test_gen.class_indices.keys())

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# ====================================
# PRECISION / RECALL / F1 TABLE
# ====================================
report = classification_report(y_true, y_pred, target_names=class_labels, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report

20/20 ━━━━━━━━━━━━━━━━━━━━ 14s 613ms/step

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.98      0.55      0.71       234
   PNEUMONIA       0.79      0.99      0.88       390

    accuracy                           0.83       624
   macro avg       0.89      0.77      0.79       624
weighted avg       0.86      0.83      0.81       624


Confusion Matrix:
 [[129 105]
 [  2 388]]


,precision,recall,f1-score,support
NORMAL,0.984733,0.551282,0.706849,234.000000
PNEUMONIA,0.787018,0.994872,0.878822,390.000000
accuracy,0.828526,0.828526,0.828526,0.828526
macro avg,0.885876,0.773077,0.792836,624.000000
weighted avg,0.861161,0.828526,0.814332,624.000000


In [24]:
from tensorflow.keras.preprocessing import image

def predict_image(model_path, img_path, class_indices):
    model = keras.models.load_model(model_path)

    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img)
    class_id = np.argmax(preds)

    inv_map = {v: k for k, v in class_indices.items()}
    return inv_map[class_id], preds